In [12]:
import json
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

verbose = 0
data_set = "gmtkn"

subset = "molecule_W4_11 molecule_G21EA molecule_G21IP molecule_DIPCS10 molecule_PA26 molecule_SIE4x4 molecule_ALKBDE10 molecule_YBDE18 molecule_AL2X6 molecule_HEAVYSB11 molecule_NBPRC molecule_ALK8 molecule_RC21 molecule_G2RC molecule_BH76 molecule_FH51 molecule_TAUT15 molecule_DC13".split(
    " "
)
subset_list = [x.split("molecule_")[1] for x in subset]
print(subset_list)

data_path_list = sorted(
    list(Path("../validate").glob(f"*{data_set}.csv")),
    key=lambda p: p.stat().st_ctime,
)
basis_args = "cc-pVDZ"
print(basis_args)

with open(f"../cc2cc/utils/{data_set}.json") as f:
    json_data = json.load(f)

# accumulate summary dictionaries for each file
summary_list = []

for i_subset in subset_list:
    summary = {
        # "subset": i_subset,
    }
    for data_path in data_path_list:

        data = pd.read_csv(data_path)
        data["name"] = data["name"].str.split(f"_{basis_args}").str[0]

        data_name = []
        data_reaction_energy_dft = []
        data_reaction_energy_ai = []

        reaction_dict = json_data[f"reaction-{i_subset}"]
        reaction_dict_copy = reaction_dict.copy()
        for i_reaction_name, i_reaction in reaction_dict_copy.items():
            systems_list = i_reaction["systems"]
            stoichiometry_list = i_reaction["stoichiometry"]

            atomic_energy_dft = 0
            atomic_energy_ai = 0
            for i in range(len(systems_list)):
                finished = True
                mole_name = f"{i_subset}-{systems_list[i]}"

                if mole_name in json_data:
                    if isinstance(json_data[mole_name], str):
                        mole_name = json_data[mole_name]
                else:
                    finished = False
                    reaction_dict.pop(i_reaction_name)
                    break

                col = data["name"] == mole_name
                if col.any():
                    atomic_energy_dft += data[col]["error_dft_ene"].values[0] * int(
                        stoichiometry_list[i]
                    )
                    atomic_energy_ai += data[col]["error_scf_ene"].values[0] * int(
                        stoichiometry_list[i]
                    )
                    if verbose == 2:
                        print(
                            data[col]["error_dft_ene"].values[0],
                            int(stoichiometry_list[i]),
                            systems_list[i],
                        )
                else:
                    finished = False
                    break

            if finished:
                data_reaction_energy_dft.append(atomic_energy_dft)
                data_reaction_energy_ai.append(atomic_energy_ai)
                data_name.append(i_reaction_name)

        data_name = np.array(data_name)
        data_reaction_energy_dft = np.array(data_reaction_energy_dft)
        data_reaction_energy_ai = np.array(data_reaction_energy_ai)

        if verbose == 1:
            dft_error_argsort = np.argsort(data_reaction_energy_dft)[:5]
            ai_error_argsort = np.argsort(data_reaction_energy_ai)[:5]

            print("====DFT error====")
            print(
                [
                    json_data[f"reaction-{i_subset}"][data_name[i]]["systems"]
                    for i in dft_error_argsort
                ]
            )
            print(data_reaction_energy_dft[dft_error_argsort])
            print("====AI error====")
            print(
                [
                    json_data[f"reaction-{i_subset}"][data_name[i]]["systems"]
                    for i in ai_error_argsort
                ]
            )
            print(data_reaction_energy_ai[ai_error_argsort])

        summary.update(
            {
                f"{data_path.stem} AI AE": f"{np.mean(np.abs(data_reaction_energy_ai) if len(data_reaction_energy_ai) else 0):.2f}",
                f"{data_path.stem} DFT AE": f"{np.mean(np.abs(data_reaction_energy_dft) if len(data_reaction_energy_dft) else 0):.2f}",
                f"{data_path.stem} Processed": f"{len(data_reaction_energy_dft)} / {len(reaction_dict)}",
            }
        )
    summary_list.append(summary)

# display one summary table for all files

header = pd.MultiIndex.from_product(
    [
        [data_path.stem for data_path in data_path_list],
        ["AI AE", "DFT AE", "Processed"],
    ],
    names=["data_path", "Type"],
)
df_summary = pd.DataFrame(
    summary_list,
    index=subset_list,
)
df_summary.columns = header
display(df_summary)

['W4_11', 'G21EA', 'G21IP', 'DIPCS10', 'PA26', 'SIE4x4', 'ALKBDE10', 'YBDE18', 'AL2X6', 'HEAVYSB11', 'NBPRC', 'ALK8', 'RC21', 'G2RC', 'BH76', 'FH51', 'TAUT15', 'DC13']
cc-pVDZ


data_path ccdft_cc-pVDZ_atom-1-2272051_gmtkn                    \
Type                                   AI AE DFT AE  Processed   
W4_11                                   2.97  29.51  140 / 140   
G21EA                                   2.14   9.75    25 / 25   
G21IP                                   0.68   1.24     1 / 36   
DIPCS10                                 0.00   0.00     0 / 10   
PA26                                    0.00   0.00     0 / 26   
SIE4x4                                  0.00   0.00     0 / 16   
ALKBDE10                                0.00   0.00      0 / 9   
YBDE18                                  0.00   0.00     0 / 18   
AL2X6                                   0.00   0.00      0 / 6   
HEAVYSB11                               0.00   0.00      0 / 7   
NBPRC                                   0.00   0.00     0 / 12   
ALK8                                    0.00   0.00      0 / 8   
RC21                                    0.00   0.00     0 / 21   
G2RC                                    0.00   0.00     0 / 25   
BH76                                    5.00   9.11    76 / 76   
FH51                                    0.00   0.00     0 / 51   
TAUT15                                  0.00   0.00     0 / 15   
DC13                                   14.65  13.09    13 / 13   

data_path ccdft_cc-pVDZ_atom-1-1464870_gmtkn                    \
Type                                   AI AE DFT AE  Processed   
W4_11                                   1.94  29.51  140 / 140   
G21EA                                   5.03   9.75    25 / 25   
G21IP                                   6.41   8.95    36 / 36   
DIPCS10                                 5.10  12.31    10 / 10   
PA26                                    5.52   2.04    19 / 26   
SIE4x4                                 19.93  21.91    16 / 16   
ALKBDE10                                4.54  18.13      9 / 9   
YBDE18                                  6.91   8.11    13 / 18   
AL2X6                                  23.66   4.15      3 / 6   
HEAVYSB11                               4.51   6.93      5 / 7   
NBPRC                                   6.47   1.52     7 / 12   
ALK8                                   18.48   3.12      7 / 8   
RC21                                    4.36   6.94     9 / 21   
G2RC                                    3.40   5.92    25 / 25   
BH76                                    3.03   9.11    76 / 76   
FH51                                    4.08   2.44     8 / 51   
TAUT15                                  1.97   2.03     8 / 15   
DC13                                   13.35  12.94     9 / 13   

data_path ccdft_cc-pVDZ_atom-1-3052180_gmtkn                    
Type                                   AI AE DFT AE  Processed  
W4_11                                   2.52  29.51  140 / 140  
G21EA                                   2.13   9.75    25 / 25  
G21IP                                   2.16   8.95    36 / 36  
DIPCS10                                 2.65  12.31    10 / 10  
PA26                                    3.49   2.22    21 / 26  
SIE4x4                                 20.35  21.91    16 / 16  
ALKBDE10                                6.72  18.13      9 / 9  
YBDE18                                  7.45   8.11    13 / 18  
AL2X6                                   5.14   4.15      3 / 6  
HEAVYSB11                               2.32   6.93      5 / 7  
NBPRC                                   2.82   1.52     7 / 12  
ALK8                                   11.02   3.12      7 / 8  
RC21                                    3.76   7.00    11 / 21  
G2RC                                    3.63   5.92    25 / 25  
BH76                                    4.62   9.11    76 / 76  
FH51                                    2.89   2.76     7 / 51  
TAUT15                                  1.58   2.03     8 / 15  
DC13                                    9.28  12.94     9 / 13

In [8]:
display(df_summary) 

data_path ccdft_cc-pVDZ_atom-1-2272051_gmtkn                    \
Type                                   AI AE DFT AE  Processed   
0                                       2.97  29.51  140 / 140   
1                                       2.14   9.75    25 / 25   
2                                       0.68   1.24     1 / 36   
3                                       0.00   0.00     0 / 10   
4                                       0.00   0.00     0 / 26   
5                                       0.00   0.00     0 / 16   
6                                       0.00   0.00      0 / 9   
7                                       0.00   0.00     0 / 18   
8                                       0.00   0.00      0 / 6   
9                                       0.00   0.00      0 / 7   
10                                      0.00   0.00     0 / 12   
11                                      0.00   0.00      0 / 8   
12                                      0.00   0.00     0 / 21   
13                                      0.00   0.00     0 / 25   
14                                      5.00   9.11    76 / 76   
15                                      0.00   0.00     0 / 51   
16                                      0.00   0.00     0 / 15   
17                                     14.65  13.09    13 / 13   

data_path ccdft_cc-pVDZ_atom-1-3052180_gmtkn                    \
Type                                   AI AE DFT AE  Processed   
0                                       2.52  29.51  140 / 140   
1                                       2.13   9.75    25 / 25   
2                                       2.16   8.95    36 / 36   
3                                       2.65  12.31    10 / 10   
4                                       3.49   2.22    21 / 26   
5                                      20.35  21.91    16 / 16   
6                                       6.72  18.13      9 / 9   
7                                       7.45   8.11    13 / 18   
8                                       5.14   4.15      3 / 6   
9                                       2.32   6.93      5 / 7   
10                                      2.82   1.52     7 / 12   
11                                     11.02   3.12      7 / 8   
12                                      3.76   7.00    11 / 21   
13                                      3.63   5.92    25 / 25   
14                                      4.62   9.11    76 / 76   
15                                      2.89   2.76     7 / 51   
16                                      1.58   2.03     8 / 15   
17                                      9.28  12.94     9 / 13   

data_path ccdft_cc-pVDZ_atom-1-1464870_gmtkn                    
Type                                   AI AE DFT AE  Processed  
0                                       1.94  29.51  140 / 140  
1                                       5.03   9.75    25 / 25  
2                                       6.41   8.95    36 / 36  
3                                       5.10  12.31    10 / 10  
4                                       5.52   2.04    19 / 26  
5                                      19.93  21.91    16 / 16  
6                                       4.54  18.13      9 / 9  
7                                       6.91   8.11    13 / 18  
8                                      23.66   4.15      3 / 6  
9                                       4.51   6.93      5 / 7  
10                                      6.47   1.52     7 / 12  
11                                     18.48   3.12      7 / 8  
12                                      4.36   6.94     9 / 21  
13                                      3.40   5.92    25 / 25  
14                                      3.03   9.11    76 / 76  
15                                      4.08   2.44     8 / 51  
16                                      1.97   2.03     8 / 15  
17                                     13.35  12.94     9 / 13

In [2]:
import numpy as np
(
    np.array([10.58423375510182, 225.43998315640636])
    - np.array([2.4392066220511355, 212.98927779812027])
)

# NBPRC-nh3-bh3

array([ 8.14502713, 12.45070536])

|File | AI AE | DFT AE | AI E | DFT E | AI Ele | DFT Ele | AI Dip | DFT Dip | Processed |
|---|---|---|---|---|---|---|---|---|---|
| atom-1-4049491 | 3.18 | 29.85 | 2.99 | 320.81 | 0.11 | 0.16 | 0.027 | 0.023 | 140 / 140 |
| atom-1-4049491 | 1.98 | 28.07 | 1.90 | 291.66 | 0.11 | 0.15 | 0.029 | 0.025 | 128 / 140 |
| atom-1-4049491 | 1.33 | 29.85 | 1.52 | 320.81 | 0.11 | 0.16 | 0.028 | 0.023 | 140 / 140 |
